In [ ]:
!pip install openai-clip -q
# Note: 'clip' package is OpenAI's official CLIP library


# CLIP & Zero-Shot Learning

OpenAI's **CLIP** (Contrastive Language-Image Pre-training) is one of the most impactful models of the last five years.
Trained on 400 million image-text pairs from the internet, it can classify images using natural language — with no task-specific training at all.

**By the end of this notebook you will:**
- Understand how contrastive pre-training aligns image and text representations
- Encode images and text into a shared embedding space
- Perform zero-shot classification on CIFAR10
- Explore prompt engineering and image-text similarity search


## 1. How CLIP Works

```
                   ┌─────────────┐
  Image ──────────>│ Image       │──> image embedding (512-d)
                   │ Encoder     │
                   │ (ViT/ResNet)│
                   └─────────────┘
                                      cosine
                                      similarity ──> contrastive loss
                   ┌─────────────┐
  Text  ──────────>│ Text        │──> text embedding (512-d)
  prompt           │ Encoder     │
                   │ (Transformer│
                   └─────────────┘
```

**Contrastive training objective:**
- For matched image-text pairs → push embeddings *together* (high cosine similarity)
- For unmatched pairs → push embeddings *apart*

**Zero-shot classification:** Encode `"a photo of a {class}"` for each class, then find the class whose text embedding is *closest* to the image embedding.


In [ ]:
import torch
import clip
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load CLIP via OpenAI (downloaded automatically)
model, preprocess = clip.load('ViT-B/32', device=device)
print(f'CLIP ViT-B/32 loaded on {device}')
print(f'Image encoder output: {model.visual.output_dim}-d')
print(f'Text  encoder output: {model.text_projection.shape[1]}-d')


## 2. Encoding Images and Text

Both images and text are projected into the same 512-dimensional space.
Similarity is measured by cosine distance — values close to 1 mean highly related.

In [ ]:
# Encode a batch of text prompts and compute similarity
cifar_classes = ['airplane','automobile','bird','cat','deer',
                 'dog','frog','horse','ship','truck']

# Text prompts for zero-shot classification
prompts = [f'a photo of a {c}' for c in cifar_classes]
text_tokens = clip.tokenize(prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)          # [10, 512]
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print(f'Text feature matrix shape: {text_features.shape}')
print(f'(10 class prompts, each a 512-d unit vector)')

# Show text-text cosine similarity to see how CLIP clusters class concepts
sim = (text_features @ text_features.T).cpu().numpy()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(10)); ax.set_xticklabels(cifar_classes, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(cifar_classes)
ax.set_title('CLIP Text-Text Cosine Similarity\n(high = semantically similar prompts)', fontsize=12)
plt.tight_layout(); plt.show()


## 3. Zero-Shot Classification on CIFAR10

No training required — just encode the class name prompts and compare to image embeddings.

In [ ]:
from torch.utils.data import DataLoader

# CIFAR10 with CLIP preprocessing (resize to 224 + CLIP normalisation)
clip_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466,0.4578275,0.40821073),
                         (0.26862954,0.26130258,0.27577711))
])
ds_test = datasets.CIFAR10('./data', train=False, download=True, transform=clip_tf)
loader  = DataLoader(ds_test, batch_size=256, shuffle=False, num_workers=2)

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)
        img_feats = model.encode_image(imgs)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        logits = (img_feats @ text_features.T) * model.logit_scale.exp()
        preds  = logits.argmax(dim=-1)
        all_preds.append(preds.cpu()); all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
acc = (all_preds == all_labels).float().mean().item()
print(f'CLIP zero-shot accuracy on CIFAR10: {acc:.4f} ({acc*100:.1f}%)')
print('(No training on CIFAR10 data — pure zero-shot!)')

# Per-class accuracy
fig, ax = plt.subplots(figsize=(10, 4))
per_class = [(all_preds[all_labels==i] == i).float().mean().item() for i in range(10)]
ax.bar(cifar_classes, per_class, color='steelblue')
ax.axhline(acc, color='red', linestyle='--', label=f'Overall: {acc:.3f}')
ax.set(title='CLIP Zero-Shot Accuracy per Class', ylabel='Accuracy')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 4. Prompt Engineering

The wording of your text prompts significantly affects CLIP's performance.

In [ ]:
# Compare different prompt templates
prompt_sets = {
    'Simple':    [f'{c}' for c in cifar_classes],
    'Photo':     [f'a photo of a {c}' for c in cifar_classes],
    'Good photo':[f'a good photo of a {c}' for c in cifar_classes],
    'Blurry':    [f'a blurry photo of a {c}' for c in cifar_classes],
    'Emoji':     [f'a {c} emoji' for c in cifar_classes],
}

results = {}
for name, prompts in prompt_sets.items():
    tokens = clip.tokenize(prompts).to(device)
    with torch.no_grad():
        t_feats = model.encode_text(tokens)
        t_feats = t_feats / t_feats.norm(dim=-1, keepdim=True)
    preds_list = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            i_feats = model.encode_image(imgs)
            i_feats = i_feats / i_feats.norm(dim=-1, keepdim=True)
            logits = (i_feats @ t_feats.T) * model.logit_scale.exp()
            preds_list.append(logits.argmax(dim=-1).cpu())
    acc_p = (torch.cat(preds_list) == all_labels).float().mean().item()
    results[name] = acc_p
    print(f'{name:<15}: {acc_p:.4f} ({acc_p*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(results.keys(), results.values(), color='coral')
ax.set(title='Zero-Shot Accuracy by Prompt Template', ylabel='Accuracy')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print('Prompt choice can shift accuracy by several percentage points!')


## 5. Exercises

1. **Ensemble prompts**: Average the text embeddings from multiple prompt templates (e.g., `'a photo of a {c}'` + `'a {c} image'` + `'a picture of a {c}'`). Does ensembling improve accuracy?
2. **Negative prompts**: Add a description for background/texture (e.g., `'a photo of a cat on grass'` vs `'a photo of a cat'`). Does adding context help or hurt?
3. **Cross-dataset**: Load CIFAR100 (100 classes). Write the prompts and compute zero-shot accuracy. How does CLIP handle 100 classes vs 10?
